# POC Extraction de données

Extraction des données provenant des rapport de G2K vers un format plus simple.

Import, définition des variables et récupération du contenue du rapport (sous forme de `string`)

In [1]:
import pandas as pd
import numpy as np
import re

## Variable global
content = ""
sections_content = {}
sections_titles = []

# Récuperaction du contenue du rapport
with open("../data/rapport-RGU_3cm.txt", "r") as f:
    content = f.read()

Définition de mes *regex*

In [30]:
section_header_pattern          = re.compile(r"^\*+$\n^\*{5}(.*)\*{5}$\n^\*+$", re.MULTILINE)
s1_kv_pattern                   = re.compile(r"^([A-ZÀ-Ÿ][a-zA-ZÀ-ÿ' ]+?)\s*:\s*(.*)$", re.MULTILINE)
s2_header_pattern               = re.compile(r"(Numéro)\s+(Début)\s+-\s+(Fin)\s+(Centroïde)\s+(Energie)\s+(FWHM)\s+(Surface)\s+(Incert\.)\s+(Fond sous)\s*\r?\n^\s*(du pic)\s+(\(canaux\))\s+(\(keV\))\s+(\(keV\))\s+(le pic)", re.MULTILINE)
s2_data_pattern                 = re.compile(r"^\s*([MmF]?)\s*(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+-\s*(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)", re.MULTILINE)
s3_header_pattern               = re.compile(r"^\s+(Nom\sdu)\s+(Indice\sde)\s+(Energie)\s+(Intensité)\s+(Activité)\s+(Incert\.)$\n^\s+(nucléide)\s+(confiance)\s+(\W\w+\W)\s+(\W%\W)\s+(\WmBq\Wg\s+\W)\s+(\WmBq\Wg\s+\W)\n", re.MULTILINE)
s3_data_pattern                 = re.compile(r"^\s*([A-Z]{1,2}-\d{1,3})?\s*(\d+\.\d+)?\s*(\d+\.\d+)(\*?)\s*(@?)\s*(\d+\.\d+)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)$", re.MULTILINE)
s4_header_1_pattern             = re.compile(r"^\s*(Nom du)\s+(Indice de)\s+(Activité moyenne)\s+(Incert\.)$\n^\s+(nucléide)\s+(confiance)\s+(pondérée)$\n^\s+(\WmBq\Wg\s+\W)\s+(\WmBq\Wg\s+\W)$", re.MULTILINE)
s4_header_2_pattern             = re.compile(r"^\s*(Numéro)\s+(Energie)\s+(Intensité)\s+(Incert\.)\s+(Type)\s+(Nucléide)$\n^\s+(du pic)\s+(\WkeV\W)\s+(\Wcoups\Wsec\W)\s+(du pic)\s+(potentiel)$", re.MULTILINE)
s4_data_1_pattern               = re.compile(r"^\s*([?X])?\s+([A-Z]{1,2}-\d{1,3})\s*(@?)\s+(\d+\.\d+)(?:\s*(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)?\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?))?", re.MULTILINE)
s4_data_2_pattern               = re.compile(r"^\s+([MmF])?\s*(\d+)\s+(\d+\.\d+)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+\.\d+)\s*(Sum|D-Esc\.|S-Esc\.|Tol\.)?\s*([A-Z]{1,2}-\d{1,3})?\s*$", re.MULTILINE)
s6_header_pattern               = re.compile(r"^\s+(.*)$\n^\s+(.*)$", re.MULTILINE)
s6_word_column_pattern          = re.compile(r"([A-Za-zÀ-ÿ]+\.?)")
s6_nucleide_line_pattern        = re.compile(r"^[+>?]\s+([A-Z]{1,2}-\d{1,3})\s+(\S+)\s+(\S+)\s+(\S+)\s+(\S+)\s+(\S+)\s+(\S+)\s+(\S+)\s+(\S+)", re.MULTILINE)


## Extraction des données de chaque section

### Setup
1 - On extrait les titre et on créer un dictionnaire de DataFrame

2- On découpe `content` par section que l'on range dans un dictionnaire `sections_content`

In [3]:
section_headers = section_header_pattern.findall(content)

# 1
for i, section in enumerate(section_headers):
    sections_titles.append(section.strip())

# 2
matches = list(section_header_pattern.finditer(content))
for i, match in enumerate(matches):
    title = match.group(1).strip()
    start = match.end()
    end = matches[i + 1].start() if i + 1 < len(matches) else len(content)
    sections_content[title] = content[start:end].strip()

### S1 - RAPPORT DE L’ANALYSE DU SPECTRE
C'est les metadata du rapport.

Pour l'instant ce n'est pas encore bon:
Mauvaise dections des clés/valeurs

In [4]:
content_s1 = sections_content[sections_titles[0]]
matches = s1_kv_pattern.findall(content_s1)
data_s1 = {key.strip(): value.strip() for key, value in matches}

####### DATA ##########
data_s1

{'Nom du fichier': 'C:\\GENIE2K\\CAMFILES\\SPECTRES\\STANDARDS\\Standard_Syrah_2024\\S',
 'Date du rapport': '02/06/2026 11:08:46',
 'Description': 'ech',
 'Identification': "Type d'échantillon          :",
 'Géométrie de mesure': 'Sensibilité de recherche    :  2.50',
 'Zone de recherche des pics': '50 - 16384',
 'Zone de calcul des surfaces': '50 - 16384',
 'Tolérance sur les énergies': '1.000 keV',
 'Date de mesure': '23/02/2024 11:38:52',
 'Temps actif': '255962.3 secondes',
 'Temps réel': '256073.1 secondes',
 'Temps mort': '0.04 %',
 'Nom de la géométrie': 'Rapport sur Les paramètres des pics  02/06/2026 11:08:46          Page  2'}

### S2 - RAPPORT ANALYSE DES PICS

In [5]:
def extract_header_s2(content):
    match = re.search(s2_header_pattern, content)
  
    if not match:
        return None
        
    columns = [
        "Marker",                                   # Marqueur (M, m, F)
        f"{match.group(1)} {match.group(9)}",       # Numéro du pic
        f"{match.group(2)} {match.group(10)}",        # Début (canaux)       
        f"{match.group(3)} {match.group(11)}",                       # Fin (canaux)
        match.group(4),                             # Centroïde
        f"{match.group(5)} {match.group(12)}",      # Energie (keV)
        f"{match.group(6)} {match.group(13)}",      # FWHM (keV)
        match.group(7),                             # Surface
        match.group(8),                             # Incert.
        f"{match.group(8)} {match.group(13)}"       # Fond sous le pic
    ]
    
    return columns



In [6]:
def extract_data_s2(content, header):
    matches = re.findall(s2_data_pattern, content)
    data = pd.DataFrame(matches, columns=header)
    data[header[1:]] = data[header[1:]].apply(pd.to_numeric, errors="coerce")
    
    return data


In [7]:
content_s2 = sections_content[sections_titles[1]]
header_s2 = extract_header_s2(content_s2)
data_s2 = extract_data_s2(content_s2, header_s2)

####### DATA ##########
data_s2

,Marker,Numéro Fond sous,Début du pic,Fin (canaux),Centroïde,Energie (keV),FWHM (keV),Surface,Incert.,Incert. (keV)
0,,1,80,97,89.94,16.21,1.27,9406.00,650.88,29580.000
1,,2,98,114,106.38,19.26,1.43,4954.00,612.65,28410.000
2,M,3,130,158,141.01,25.67,0.85,4291.00,269.06,23080.000
3,m,4,130,158,150.25,27.38,0.86,3786.00,266.29,24920.000
4,,5,191,205,200.49,36.68,0.50,591.10,525.79,23780.000
...,...,...,...,...,...,...,...,...,...,...
181,,182,15550,15569,15559.19,2879.40,0.28,23.17,15.24,9.833
182,,183,15777,15796,15786.26,2921.43,1.21,25.69,18.45,16.310
183,,184,15872,15891,15881.97,2939.14,0.35,19.00,12.08,5.000
184,,185,16081,16102,16091.77,2977.97,0.84,24.92,13.29,5.083


### S3 - RAPPORT IDENTIFICATION DES NUCLEIDES

In [ ]:
def extract_header_s3(content):
    matches = re.search(s3_header_pattern, content)

    if not matches:
        return None
    
    columns = [
        f"{matches.group(1)} {matches.group(7)}",
        f"{matches.group(2)} {matches.group(8)}",
        f"{matches.group(3)} {matches.group(9)}",
        "Marker *",
        "Marker @",
        f"{matches.group(4)} {matches.group(10)}",
        f"{matches.group(5)} {matches.group(11)}",
        f"{matches.group(6)} {matches.group(12)}"
    ]
   
    return columns

In [ ]:
def extract_data_s3(content, header):
    matches = re.findall(s3_data_pattern, content)
    df = pd.DataFrame(matches ,columns=header)
    
    numeric = ['Indice de confiance', 'Energie (keV)', 'Intensité (%)', 'Activité (mBq/g   )', 'Incert. (mBq/g   )']
    df[numeric] = df[numeric].apply(pd.to_numeric, errors="coerce") 
    
    return df

In [10]:
content_s3      = sections_content[sections_titles[2]]
header_s3       = extract_header_s3(content_s3)
data_s3         = extract_data_s3(content_s3, header_s3)

### S4 - RAPPORT IDENTIFICATION AVEC CORRECTION D’INTERFERENCE

In [ ]:
def extract_header_1_s4(content):
    matches = re.search(s4_header_1_pattern, content)
    if not matches:
        return None
    
    columns = [
        "Marker (X/?)",
        f"{matches.group(1)} {matches.group(5)}",
        "Marker (@)",
        f"{matches.group(2)} {matches.group(6)}",
        f"{matches.group(3)} {matches.group(7)} {matches.group(8)}",
        f"{matches.group(4)} {matches.group(9)}"
    ]
    
    return columns

In [ ]:
def extract_header_2_s4(content):
    matches = re.search(s4_header_2_pattern, content)
    if not matches:
        return None
    
    columns = [
        "Marker (M/m/F)",
        f"{matches.group(1)} {matches.group(7)}",
        f"{matches.group(2)} {matches.group(8)}",
        f"{matches.group(3)} {matches.group(9)}",
        f"{matches.group(4)}",
        f"{matches.group(5)} {matches.group(10)}",
        f"{matches.group(6)} {matches.group(11)}"
    ]
    
    return columns

In [ ]:
def extract_data_1_s4(content, header):
    pass

In [ ]:
def extract_data_2_s4(content, header):
    pass

In [ ]:
content_s4 = sections_content[sections_titles[3]]
header_1_s4 = extract_header_1_s4(content_s4)
header_2_s4 = extract_header_2_s4(content_s4)

data_1_s4 = None
data_2_s4 = None

print(header_1_s4)
print(header_2_s4)

['Marker (X/?)', 'Nom du nucléide', 'Marker (@)', 'Indice de confiance', 'Activité moyenne pondérée (mBq/g   )', 'Incert. (mBq/g   )']
['Marker (M/m/F)', 'Numéro du pic', 'Energie (keV)', 'Intensité (coups/sec)', 'Incert.', 'Type du pic', 'Nucléide potentiel']


### S5 - RAPPORT LIMITES DE DETECTION

In [13]:
content_s5 = sections_content[sections_titles[4]]

### S6 - RAPPORT LIMITES DE DETECTION  ISO 11929

In [14]:
def extract_header_s6(content):
    header = []
    matches = re.findall(s6_header_pattern, content)

    l1 = re.findall(s6_word_column_pattern, matches[0][0])
    l2 = re.findall(s6_word_column_pattern, matches[0][1])
    
    # Fusion en partant de la fin
    for a, b in zip(reversed(l1), reversed(l2)):
        header.insert(0, f"{a} {b}".strip())

    # Si l1 est plus longue, on conserve le début restant
    if len(l1) > len(l2):
        header = l1[: len(l1) - len(l2)] + header

    return header

In [15]:
def extract_data_s6(content, header):
    matches = re.findall(s6_nucleide_line_pattern, content)
    data = pd.DataFrame(matches, columns=header)
    data[header[1:]] = data[header[1:]].apply(pd.to_numeric, errors="coerce") 
    
    return data

In [16]:
content_s6 = sections_content[sections_titles[5]]
header_s6 = extract_header_s6(content_s6)
data_s6 = extract_data_s6(content_s6, header_s6)

####### DATA ##########
data_s6

,Nucléide,LD,SD,Limite Basse,Limite Haute,Moyenne Activité,pondérée Incert.,Meilleure Activité,Estimation Incert.
0,PB-210,70.50,35.20,4880.0,5040.0,4960.0,41.60,4960.0,83.10
1,BI-214,17.20,8.55,4790.0,4910.0,4850.0,29.70,4850.0,59.50
2,PB-214,7.51,3.74,4700.0,4770.0,4730.0,18.50,4730.0,37.00
3,RA-226,214.00,107.00,5670.0,6590.0,6130.0,234.00,6130.0,467.00
4,TH-227,15.00,7.47,186.0,206.0,196.0,5.20,196.0,10.40
5,TH-230,507.00,249.00,4930.0,6790.0,5860.0,475.00,5860.0,949.00
6,PA-231,29.70,14.70,138.0,167.0,152.0,7.38,152.0,14.80
7,TH-234,64.60,28.70,3620.0,8280.0,5950.0,1190.00,5950.0,2380.00
8,U-235,9.51,4.66,231.0,245.0,238.0,3.55,238.0,7.10
9,AM-241,5.87,2.77,17.0,30.6,23.8,3.46,23.8,6.93


# Zone de Test